In [ ]:
# @title Iron Condor Scanner (Final: Clean Table + Credit Rules)
# @markdown ### 🦅 Strategy: High Premium + Statistical Edge
# @markdown 1. **Filter:** Range Bound (Strict Technicals) + Safe (No Earnings).
# @markdown 2. **Selection:** Picks the **Top 30 Highest IV** stocks.
# @markdown 3. **Ranking:** Re-sorts those 30 by **Edge** (IV / HV) to show the best statistical setups first.

import yfinance as yf
import pandas as pd
import numpy as np
import datetime
from datetime import timedelta
import os
import pickle
import warnings
import logging

# Mute warnings
warnings.filterwarnings("ignore")
logging.getLogger('yfinance').setLevel(logging.CRITICAL)

# --- 1. CONFIGURATION ---
TARGET_DTE = 45          
EARNINGS_BUFFER = 60     
MAX_RESULTS = 30         
CACHE_FILE = "market_data_cache.pkl"

# Master List
tickers_raw = "A AA AAL AAP AAPL ABBV ABNB ABT ACN ADBE ADI ADM ADP ADSK AEP AES AFL AFRM AGNC AI AIG AKAM ALB ALK ALL AMAT AMD AMGN AMRN AMZN APA APH APTV AVGO AXP BA BABA BAC BAX BBY BEN BIDU BIIB BITO BK BKR BMY BP BSX BX BYND C CAH CAT CB CCI CCJ CCL CF CFG CHWY CI CL CLF CLX CMCSA CME CNC CNP COF COIN COP COST CPB CPRI CRM CRON CRWD CSCO CSX CTVA CVNA CVS CVX CZR D DAL DD DE DELL DHI DHR DIA DIS DKNG DLR DOCU DOW DRI DVN DXC EA EBAY ED EEM EFA EIX EL EMR ENPH EOG EQR EQT ETSY EVRG EW EWJ EWW EWY EWZ EXC EXPE F FANG FAST FCX FDX FE FI FITB FOXA FSLR FTI FTV FXE FXI GD GDX GE GILD GLD GLW GM GME GOOG GOOGL GPRO GPS GS HAL HBAN HBI HCA HD HIG HLT HOG HOLX HON HPE HPQ HYG IBB IBIT IBM ICE IEF INTC IP IPG IRM IVZ IWM IYR JCI JD JETS JNJ JNPR JPM K KHC KMI KO KR KRE KSS KWEB LEN LI LKQ LLY LNC LOW LQD LUMN LUV LVS LYB LYFT MA MAR MARA MAS MCD MCHP MDLZ MDT MET META MGM MMM MO MOS MPC MRK MRNA MRVL MS MSFT MSTR MTB MTCH MU NCLH NEE NEM NET NFLX NKE NOW NRG NTAP NTES NVAX NVDA NWL NWSA O OIH OKE OMC ON ORCL OXY PARA PBR PDD PENN PEP PFE PFG PG PGR PINS PLTR PNC PPL PRU PSX PTON PYPL QCOM QQQ RBLX RCL RF RIG RIOT RIVN ROKU ROST RTX RUN SBUX SCHD SCHW SEDG SHOP SIRI SLB SLV SMCI SMH SNAP SNOW SO SOFI SOXL SOXS SPG SPX SPXL SPXS SPY SQQQ STX SWKS SYF SYY T TAP TBT TCOM TDOC TFC TGT TJX TLT TMO TMUS TPR TQQQ TRIP TRV TSLA TSM TSN TT TTD TTWO TXN U UA UAA UAL UBER ULTA UNG UNH UNP UPS UPST URBN USB USO UVXY V VALE VFC VGK VTR VXX VZ WAB WBA WBD WDC WFC WM WMB WMT WU WY WYNN X XBI XEL XHB XLB XLC XLE XLF XLI XLK XLP XLRE XLU XLV XLY XOM XOP XRT XRX XSP XYZ YELP YINN ZION ZM"
ticker_list = list(set(tickers_raw.split()))

# --- 2. CACHING ENGINE ---
def get_market_data(tickers):
    today_str = datetime.date.today().strftime("%Y-%m-%d")
    if os.path.exists(CACHE_FILE):
        try:
            with open(CACHE_FILE, 'rb') as f:
                cache = pickle.load(f)
            if cache['data'] is not None and not cache['data'].empty:
                print(f"💾 Cache Found. Loading data...")
                return cache['data']
        except: pass
    
    print(f"⏳ Downloading fresh data...")
    data = yf.download(tickers, period="6mo", group_by='ticker', threads=True, progress=False)
    with open(CACHE_FILE, 'wb') as f: pickle.dump({'date': today_str, 'data': data}, f)
    return data

# --- 3. ROBUST MATH FUNCTIONS ---
def calc_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).ewm(alpha=1/period, adjust=False).mean()
    loss = (-delta.where(delta < 0, 0)).ewm(alpha=1/period, adjust=False).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def calc_bollinger(series, period=20, std_dev=2):
    sma = series.rolling(window=period).mean()
    std = series.rolling(window=period).std()
    return sma + (std * std_dev), sma - (std * std_dev)

def calc_adx_robust(df, period=14):
    local_df = df.copy()
    local_df['H-L'] = local_df['High'] - local_df['Low']
    local_df['H-PC'] = abs(local_df['High'] - local_df['Close'].shift(1))
    local_df['L-PC'] = abs(local_df['Low'] - local_df['Close'].shift(1))
    local_df['TR'] = local_df[['H-L', 'H-PC', 'L-PC']].max(axis=1)
    
    local_df['+DM'] = np.where((local_df['High'] - local_df['High'].shift(1)) > (local_df['Low'].shift(1) - local_df['Low']), 
                               np.maximum(local_df['High'] - local_df['High'].shift(1), 0), 0)
    local_df['-DM'] = np.where((local_df['Low'].shift(1) - local_df['Low']) > (local_df['High'] - local_df['High'].shift(1)), 
                               np.maximum(local_df['Low'].shift(1) - local_df['Low'], 0), 0)

    local_df['TR_S'] = local_df['TR'].ewm(alpha=1/period, adjust=False).mean()
    local_df['+DM_S'] = local_df['+DM'].ewm(alpha=1/period, adjust=False).mean()
    local_df['-DM_S'] = local_df['-DM'].ewm(alpha=1/period, adjust=False).mean()

    local_df['+DI'] = 100 * (local_df['+DM_S'] / local_df['TR_S'])
    local_df['-DI'] = 100 * (local_df['-DM_S'] / local_df['TR_S'])
    local_df['DX'] = 100 * abs(local_df['+DI'] - local_df['-DI']) / (local_df['+DI'] + local_df['-DI'])
    local_df['ADX'] = local_df['DX'].ewm(alpha=1/period, adjust=False).mean()
    return local_df['ADX']

def calc_hv(series, window=30):
    log_returns = np.log(series / series.shift(1))
    return log_returns.rolling(window=window).std() * np.sqrt(252)

# --- 4. SCANNER (CASCADING) ---
def scan_technicals(data, tickers, adx_lim, rsi_min, rsi_max):
    candidates = []
    print(f"   👉 Filtering: ADX < {adx_lim} | {rsi_min} < RSI < {rsi_max} | Inside BB")
    
    for ticker in tickers:
        try:
            try: df = data[ticker].copy().dropna()
            except: continue
            if len(df) < 50: continue

            df['RSI'] = calc_rsi(df['Close'])
            df['ADX'] = calc_adx_robust(df)
            df['HV']  = calc_hv(df['Close'])
            upper, lower = calc_bollinger(df['Close'])
            
            curr_rsi = df['RSI'].iloc[-1]
            curr_adx = df['ADX'].iloc[-1]
            curr_hv  = df['HV'].iloc[-1]
            close = df['Close'].iloc[-1]
            u_bb = upper.iloc[-1]
            l_bb = lower.iloc[-1]
            
            if np.isnan(curr_adx) or np.isnan(curr_rsi): continue

            if (curr_adx < adx_lim) and (rsi_min < curr_rsi < rsi_max) and (l_bb < close < u_bb):
                candidates.append({
                    'Symbol': ticker, 'Price': close, 'ADX': curr_adx, 'RSI': curr_rsi, 'HV': curr_hv
                })
        except: continue
    return candidates

# --- 5. FUNDAMENTALS ---
def check_iv_earnings(tech_list):
    final_list = []
    print(f"⚡ Checking IV & Earnings for {len(tech_list)} survivors...")
    
    for i, item in enumerate(tech_list):
        if i % 10 == 0: print(f"   Processing {i}/{len(tech_list)}...", end="\r")
        tk = yf.Ticker(item['Symbol'])
        
        # EARNINGS
        next_earn = "None/ETF"
        safe = True
        try:
            dates = tk.earnings_dates
            if dates is not None and not dates.empty:
                future = dates[dates.index > pd.Timestamp.now()]
                if not future.empty:
                    next_date = future.index[0]
                    next_earn = next_date.strftime('%Y-%m-%d')
                    if (next_date - pd.Timestamp.now()).days < EARNINGS_BUFFER:
                        safe = False
        except: pass
        if not safe: continue

        # IV CHECK
        try:
            exps = tk.options
            if not exps: continue
            today = datetime.date.today()
            valid_exps = [e for e in exps if 30 <= (datetime.datetime.strptime(e, "%Y-%m-%d").date() - today).days <= 60]
            if not valid_exps: continue
            
            target = today + timedelta(days=TARGET_DTE)
            expiry = min(valid_exps, key=lambda x: abs((datetime.datetime.strptime(x, "%Y-%m-%d").date() - target).days))
            
            opt = tk.option_chain(expiry)
            calls = opt.calls; puts = opt.puts
            cp = item['Price']
            
            atm_c = calls.iloc[(calls['strike'] - cp).abs().argsort()[:1]]
            atm_p = puts.iloc[(puts['strike'] - cp).abs().argsort()[:1]]
            iv_c = atm_c['impliedVolatility'].values[0] if not atm_c.empty else 0
            iv_p = atm_p['impliedVolatility'].values[0] if not atm_p.empty else 0
            avg_iv = (iv_c + iv_p) / 2
            
            if avg_iv == 0: continue
            
            item['IV'] = avg_iv
            item['Next Earnings'] = next_earn
            
            # EDGE CALC
            hv = item['HV']
            edge = avg_iv / hv if hv > 0 else 0
            item['Edge'] = edge
            
            final_list.append(item)
        except: continue
        
    return final_list

# --- 6. EXECUTION ---
market_data = get_market_data(ticker_list)

scans = [
    {"name": "STRICT", "adx": 30, "rsi_min": 40, "rsi_max": 60},
    {"name": "MODERATE", "adx": 45, "rsi_min": 35, "rsi_max": 65},
    {"name": "RELAXED", "adx": 55, "rsi_min": 30, "rsi_max": 70}
]

tech_pass = []
used_mode = "NONE"

print("\n🔍 STARTING SCAN...")

for scan in scans:
    print(f"🔹 Attempting {scan['name']} Mode...")
    results = scan_technicals(market_data, ticker_list, scan['adx'], scan['rsi_min'], scan['rsi_max'])
    if len(results) >= 10: 
        tech_pass = results
        used_mode = scan['name']
        print(f"✅ Found {len(results)} matches using {scan['name']} filters.")
        break
    else:
        print(f"⚠️ Only {len(results)} matches. Relaxing filters...")

if not tech_pass:
    print("❌ No stocks passed even Relaxed filters.")
else:
    final_results = check_iv_earnings(tech_pass)
    
    if final_results:
        # STEP 1: SORT BY IV FIRST (Get Top 30 High Premiums)
        df_all = pd.DataFrame(final_results).sort_values(by='IV', ascending=False)
        
        # STEP 2: KEEP ONLY TOP 30
        df_top30 = df_all.head(MAX_RESULTS).copy()
        
        # STEP 3: SORT THE FINAL 30 BY EDGE (Quality Sort)
        df_final = df_top30.sort_values(by='Edge', ascending=False).reset_index(drop=True)
        
        df_final.index += 1 
        df_final.index.name = 'Rank'
        df_final = df_final.reset_index()

        # OUTPUT
        print("\n" + "="*80)
        print(f"🏆 TOP {len(df_final)} IRON CONDOR CANDIDATES")
        print(f"Logic: Top 30 High IV Stocks -> Re-sorted by Edge")
        print("="*80)
        
        disp = df_final.copy()
        disp['IV'] = disp['IV'].apply(lambda x: f"{x:.1%}")
        disp['HV'] = disp['HV'].apply(lambda x: f"{x:.1%}")
        disp['Edge'] = disp['Edge'].apply(lambda x: f"{x:.2f}x")
        disp['Price'] = disp['Price'].apply(lambda x: f"${x:.2f}")
        
        # Clean Display
        print(disp[['Rank', 'Symbol', 'Price', 'IV', 'HV', 'Edge', 'Next Earnings']].to_string(index=False))

        print("\n" + "="*30 + " TRADINGVIEW LIST " + "="*30)
        print(",".join(df_final['Symbol'].tolist()))
    else:
        print("❌ Survivors failed IV/Earnings check.")

# --- 7. TRADE INSTRUCTIONS ---
print("\n" + "="*80)
print("🦅 TRADE ENTRY RULES")
print("="*80)
print(f"""
1. LIST LOGIC:
   - Rank #1 = High Premium + Strongest Statistical Edge (Safe bet).
   - Rank #30 = High Premium + Weaker Statistical Edge (Riskier bet).

2. SETUP ({TARGET_DTE} DTE):
   - Sell Call: Delta 0.16 - 0.20 (Resistance)
   - Sell Put:  Delta 0.16 - 0.20 (Support)
   - Wings: Equal width ($5 or $10).

3. MINIMUM CREDIT RULE (The 1/3 Rule):
   - You must collect roughly **33% (1/3)** of the wing width.
   - Example ($5.00 Wings): Credit must be at least **$1.65**.
   - Example ($10.00 Wings): Credit must be at least **$3.30**.
   - If the market offers less than this, the risk/reward is poor. SKIP IT.

4. MANAGEMENT:
   - Profit: 50% of credit.
   - Stop Loss: 2x credit received.
""")